# TradeSense · Component 03 · Layer 3 v3 builder (Protocol B2, step 1)

Builds the synthetic-event data sets for the second round of the bar-level models, **after freezing the protocol**:

1. the protocol text is hashed and stored (the freeze; a second run with different text stops);
2. the generator library is written and hashed, its 11 unit tests and a reconstruction test on the real Layer 1 matrices must pass;
3. four injected sets per symbol are built: TRAIN-3 (Layer 1 train bars), VAL-3 (Layer 1 validation + former test split, 27 days), CAL-3 (Aug 2 to Aug 31), TEST-3 (Sep 1 to Oct 31). Events are injected on the raw bar quantities and the six features are recomputed with the frozen contract. Eight families: the six v1 families plus two unseen families (momentum ignition, slow accumulation) that never appear in TRAIN-3;
4. a detectability sanity check (a simple robust-z score) and a manifest with sha256 of every output.

**Does not train anything and does not look at TEST-3 results.** CPU, about 10 to 20 minutes. Output folder: `layer3_v3/`. Existing files are only read.

In [ ]:
import os, json, glob, time, hashlib, warnings, datetime
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets')); assert BASE.is_dir(), BASE
OUT = BASE / 'layer3_v3'; OUT.mkdir(exist_ok=True)
SYMBOLS = ['BTCUSDT', 'ETHUSDT']
LINES = []; T0 = time.time()
def say(s=''): print(s); LINES.append(str(s))
def sha_file(p, n=16): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:n]
def find1(pat):
    c = [x for x in glob.glob(str(BASE/'**'/pat), recursive=True) if 'backup' not in x.lower()]; assert c, pat; return sorted(c)[0]

Mounted at /content/drive


## 1 · Freeze the protocol

In [ ]:
PROTOCOL = r'''
# TradeSense C03 – Protocol B2 (bar-level models, second round: AE-ctx, XGBoost, Transformer)
Status: APPROVED 2026-10-14 (Rashmi delegated all open choices: "do what will give the best results"). It is frozen when notebook 27 hashes this exact text (sha256 stored in Drive) before any Layer 3 v3 file, normalisation statistic, model or score of this round exists. Supersedes nothing: v1 models (AE v1/v2, XGBoost v1, Transformer v1) stay untouched and are re-scored as comparators.

## 1. Why a second round
1. Layer 1 is validated against raw trades (reconciliation, notebooks 23–24); the v1 models were trained on corrected data. Retraining is NOT needed for data errors.
2. Drift: rush_order_density (trades per unit volume) rises steadily (BTC weekly median ~50 in May, ~75 in July, ~130 late Aug, ~200 Sep–Oct; ETH 3 → 6 → 8 → 12; trades per bar ×3, volume per bar flat). Source check passed (Tardis = Binance trade files). Estimated shift of the Aug–Oct median against the v1 scaler: +1.6 SD (BTC), +2.2 SD (ETH), October +2.2 / +3.1 SD. Thresholds fixed on Layer 1 validation are therefore expected to over-alarm in Aug–Oct (an expectation, measured below, not a result).
3. Evidence base is thin: ~30 validation / 36 test events per symbol, thresholds from ~13 validation days, windows of 16 bars only, Layer 3 labels injected in feature space with no cross-feature coherence guarantee.
4. Known weak families: volume burst, trade-flow imbalance, wash-like (XGBoost leave-one-family-out 5/34, 7/32, 2/32).

## 2. Data and roles (all frozen by sha256 before use)
- Layer 1 (processed_layer1_final_v1, 90 days 2024-05-01..07-29): TRAIN = first 63 d (bars 0..90719). DEV = the v1 validation (13 d) and the v1 test split (14 d) together, 27 d (bars 90720..129599); the v1 test split was already examined by v1, so it is used here as additional development data, never as a test. All selection, early stopping and the burden quantiles q_b use DEV only. Warm-up: the first 15 bars of every continuous run have volume_acceleration = 0 by contract (the stored matrix does not mask bars 1..14; this round masks them) and are excluded from training, thresholds and evaluation together with bars < 30.
- Aug–Oct 2024 bars (notebook 25, individual-trade source, sha256 of both parquet files recorded):
  - CAL = 2024-08-02..2024-08-31 (30 days, 08-01 excluded because it is a TGNN test day). Used only to measure drift and report realised alarm rates. No model selection on CAL.
  - TEST-B = 2024-09-01..2024-10-31 (61 days). Opened once (section 9).
  - 2024-08-01 is a TGNN/fusion test day (book data exist); its bars are never used for any fitting or selection.
- Layer 3 v3 (new, section 3): synthetic events injected on RAW bars, features recomputed by the frozen contract, so cross-feature coherence holds by construction. Separate injected sets: TRAIN-3 (on TRAIN bars), VAL-3 (on DEV bars), CAL-3 (on CAL bars), TEST-3 (on TEST-B bars). Each set is built as independent replicas of the same genuine bars (at most 50% of the bars inside events, gaps and tails per replica, own seed per replica); replicas are pooled to the event totals below.
- Layer 2 real labelled data (Sapienza, Mt.Gox) remain external real validation (reported separately, domain shift stated) in D3, not in this protocol.

## 3. Layer 3 v3 generator (written and hashed before use; own seed per set)
- Families: the six v1 families (pump_and_dump_like, fake_breakout_like, abnormal_volume_burst, trade_flow_imbalance, wash_like_aggregate_proxy, liquidity_stress_proxy) plus two NEW families used only as unseen families: momentum_ignition_like (multi-bar one-sided flow ramp with rising volume and trades, then reversal) and slow_accumulation_like (low-amplitude one-sided flow over 30–120 bars, long duration).
- Mechanics act on bar fields (buy/sell volume, volume, trade count, large-trade count, high/low/close path) and the six features are recomputed. Each family has a fixed recipe, amplitude scale drawn log-uniform 0.5–2.0 × base, duration drawn from a family range, events ≥ 60 bars apart, dependency tail bars labelled and excluded as in v2.
- Events per symbol (pooled over replicas): TRAIN-3 600, VAL-3 300, CAL-3 200, TEST-3 300 (about 10× the v2 counts). The injection works on the raw bar quantities behind the six features (bar log-return, high–low range ratio, volume, buy share, trade count, large-trade count); price levels are not rebuilt because no feature uses them. The two new families are generated in VAL-3, CAL-3 and TEST-3 only, never in TRAIN-3.
- Negative definition as v1: clean bar = no non-clean bar in the model's input window (window length W of the model, so long windows have fewer negatives).

## 4. Normalisation options (both pre-registered, compared)
- N0 (v1): median/(IQR/1.349) fitted on Layer 1 TRAIN bars ≥ 15, log1p on rush_order_density and large_trade_count, clip ±20.
- N1 (causal trailing): same transforms, but median and scale of each feature come from the previous 7 days (10,080 bars, minimum 1,440; before that N0 statistics), computed only from bars strictly before t on the observed series (injected bars included, as in deployment); scale floor = 0.1 × N0 scale.
Decision rule (validation only): N1 is adopted for a model family if it is non-inferior on VAL-3 (H-B2) AND reduces the alarm-rate inflation measured on CAL (H-B1); otherwise N0.

## 5. Models (3 seeds each: 11, 22, 33; v1 selection tolerances kept)
- AE-ctx: the v2 conditional autoencoder design (current bar + causal context mean/std of previous 15 bars, bottleneck 3), retrained under each normalisation; score = current-bar error with fmax (and fmean reported). Existing v1/v2 AE files are re-scored as comparators.
- XGBoost v2: feature sets A (rolling features, 53) and B (A + AE columns). For B the AE columns of TRAIN bars are OUT-OF-FOLD (5 time-ordered blocks, 60-bar purge gap; each block scored by an AE trained on the other blocks); validation/test bars use the AE trained on all train bars. Candidates: set {A,B} × scope {pooled, per_symbol} × depth {3,5}; hyper-parameters as v1.
- Transformer v2: window W ∈ {16, 32, 64} × (d_model, layers) ∈ {(32,1), (64,1), (64,2)} = 9 candidates, otherwise as v1 (pre-norm, 4 heads, BCE pos_weight 5, AdamW, patience 5). Negatives require a clean window of length W.
- Comparators declared: v1 XGBoost, v1 Transformer, AE v2 recommended (all re-scored on Layer 3 v3), robust-z baseline rz.
- Fit data: TRAIN-3 rows only. Early stopping and selection: VAL-3 rows only. CAL-3 and TEST-3 never used for fitting or selection.

## 6. Thresholds and burden
- Burden definition as v1: alarm episodes per day on genuine bars (alarm bars merged when gap ≤ 5 bars); budgets 1, 2, 4 per day, primary 2.
- T0 static: threshold from DEV genuine bars (v1 rule). Used as the drift stress test.
- T1 rolling (the deployable rule): threshold_t = q_b-quantile of the model's own scores in the previous 7 days (strictly before t), q_b chosen on DEV genuine bars per budget and then frozen. No labels are used.
- Realised episodes per day on genuine CAL and TEST-B bars are always reported next to recall, for T0 and T1.

## 7. Hypotheses (pre-registered, reported as failed if they fail)
- H-B1 (drift): on genuine CAL bars, T0 alarm rates of the N0 models exceed 2× the budget for at least one final model; N1 with T1 reduces this inflation by ≥ 50% for every adopted model and keeps realised episodes/day within [0.5×, 2×] of the budget.
- H-B2 (non-inferiority): N1 recall at the primary budget on VAL-3, macro over families, averaged over BTC/ETH, is not more than 0.05 below N0.
- H-B3 (window): the selected Transformer window > 16 improves leave-one-family-out macro recall (six v1 families) by ≥ 0.05 over the best W = 16 candidate; otherwise W = 16 stays.
- H-B4 (out-of-fold AE): XGBoost set B (out-of-fold) beats set A by ≥ 0.03 validation recall at the primary budget; otherwise set A.
- H-B5 (unseen families): on the two new families (never trained), the best trained model has recall at the primary budget above rz by ≥ 0.05; otherwise it is described as family-specific.
- H-B6 (improvement): final v2 models beat the v1 models re-scored on TEST-3 by ≥ 0.05 macro recall at the primary budget with T1 realised episodes/day within [0.5×, 2×] of the budget. Event-bootstrap 95% intervals are reported for all recall numbers.

## 8. Fixed checks (unit tests before any claim)
Protocol hash intact and frozen before data load; frozen inputs unchanged (sha256); preprocessing and N1 statistics causal (changing bars after t leaves values ≤ t unchanged); injected events touch only event bars and their declared tails; clean-negative windows identical in genuine and injected series; fitting rows inside TRAIN-3, early-stopping rows inside VAL-3; OOF AE columns never come from an AE that saw that block; T0/T1 thresholds recomputed independently; recall recomputed by a plain loop; same-seed retraining reproduces scores (Spearman ≥ 0.98); TEST-B and TEST-3 opened exactly once after validation and CAL results are saved; manifest with sha256 of every output.

## 9. Test discipline
Order: freeze protocol → build Layer 3 v3 sets (seeds fixed, hashes recorded) → train/select on TRAIN-3/VAL-3 → measure CAL (genuine alarm rates, CAL-3 recall) and save → declare the final set → open TEST-B/TEST-3 once. No model, threshold rule or normalisation changes after CAL is saved. The TGNN/fusion test days (2024-08-01, 09-01, 10-01) are opened separately in D3 with the thresholds fixed here.

## 10. Limitations stated in advance
Labels are synthetic mechanisms, not verified manipulation; CAL and TEST-B genuine bars are unlabelled (alarms are not proven false alarms); the cause of the drift is unknown; two symbols, 6 months of bars; T1 depends on the clean-dominant assumption of the trailing window; GPU quota limits the Transformer grid (9 candidates, resumable).

## 11. Build plan (notebooks)
27 Layer 3 v3 builder (+ hash) · 28 normalisation N0/N1 and AE-ctx (with OOF folds) · 29 XGBoost v2 · 30 Transformer v2 (GPU) · 31 CAL measurement · 32 TEST opening and validation tests.
'''

ph = hashlib.sha256(PROTOCOL.encode()).hexdigest()
pf = OUT/'protocol_bar_models_v2.md'; hf = OUT/'protocol_bar_models_v2.sha256'
if hf.exists():
    assert hf.read_text().split()[0] == ph, 'FROZEN protocol differs from the stored hash: stop (never change a frozen protocol silently)'
    say('protocol already frozen; hash matches: ' + ph[:16])
else:
    pf.write_text(PROTOCOL); hf.write_text(ph + '  frozen ' + datetime.datetime.utcnow().isoformat() + 'Z\n')
    say('PROTOCOL FROZEN now; sha256 ' + ph)

PROTOCOL FROZEN now; sha256 443f99d5faf5929f9b58d510e5aaf955a65359ef577c9590c54246d3d7254e9e


## 2 · Generator library and unit tests

In [ ]:
open("l3v3_lib.py", "w").write(r'''
"""TradeSense C03 - Layer 3 v3 generator (Protocol B2). Injects synthetic manipulation-like mechanisms on the RAW bar
quantities behind the six Layer 1 features (bar log-return, high-low ratio, volume, buy volume, trade count, large-trade
count) and recomputes the six features with the frozen contract, so cross-feature coherence holds by construction."""
import numpy as np, pandas as pd, hashlib, json

FAMS = {1: 'pump_and_dump_like', 2: 'fake_breakout_like', 3: 'abnormal_volume_burst', 4: 'trade_flow_imbalance',
        5: 'wash_like_aggregate_proxy', 6: 'liquidity_stress_proxy', 7: 'momentum_ignition_like', 8: 'slow_accumulation_like'}
OLD_FAMS = [1, 2, 3, 4, 5, 6]; NEW_FAMS = [7, 8]
DUR = {1: (8, 30), 2: (6, 20), 3: (1, 8), 4: (10, 60), 5: (10, 60), 6: (5, 25), 7: (15, 45), 8: (30, 120)}
TAIL, GAP, LOOK, MASK = 15, 45, 120, 15
FLOOR = {'BTCUSDT': 2.5399, 'ETHUSDT': 27.7658}
FEATS = ['log_returns', 'trade_flow_imbalance', 'hl_spread_ratio', 'rush_order_density', 'volume_acceleration', 'large_trade_count']


def raw_from_matrix(m):
    """raw quantities from a Layer 1 style matrix (columns volume, trades_count, large_trade_count + the six features)."""
    vol = m['volume'].to_numpy(float); tfi = m['trade_flow_imbalance'].to_numpy(float)
    buy = (vol + tfi * (vol + 1e-8)) / 2.0                      # exact inverse of (buy-sell)/(buy+sell+1e-8)
    return {'r': m['log_returns'].to_numpy(float).copy(), 'hl': m['hl_spread_ratio'].to_numpy(float).copy(), 'vol': vol.copy(),
            'buy': buy.copy(), 'trades': m['trades_count'].to_numpy(float).copy(), 'large': m['large_trade_count'].to_numpy(float).copy()}


def features_from_raw(raw, symbol):
    v = raw['vol']; n = len(v)
    out = pd.DataFrame({'log_returns': raw['r'], 'trade_flow_imbalance': (2 * raw['buy'] - v) / (v + 1e-8), 'hl_spread_ratio': raw['hl'],
                        'rush_order_density': raw['trades'] / (v + 1e-8)})
    va = np.zeros(n)
    if n > MASK:   # exact per-window statistics (bar t uses bars t-15..t-1) so untouched windows stay bit-identical
        w = np.lib.stride_tricks.sliding_window_view(v, 15)[: n - 15]      # window k = v[k:k+15] -> bar k+15
        mu = w.mean(axis=1); sd = w.std(axis=1, ddof=1); va[15:] = (v[15:] - mu) / np.maximum(sd, FLOOR[symbol])
    out['volume_acceleration'] = va
    out['large_trade_count'] = raw['large']
    return out


def _smooth(D):
    u = (np.arange(D) + 0.5) / D
    return u, np.minimum(1.0, np.minimum(u * 5, (1 - u) * 5))


def apply_event(raw, fam, t0, D, a, rng):
    """modify raw quantities in [t0, t0+D) in place; baselines from the previous LOOK bars of the ORIGINAL series (causal)."""
    sl = slice(t0, t0 + D); b0 = slice(t0 - LOOK, t0)
    r0 = raw['r'][b0]; sig = 1.4826 * np.median(np.abs(r0 - np.median(r0))) + 1e-9
    k = np.arange(D); u, ramp = _smooth(D); sign = rng.choice([-1.0, 1.0]); amin = min(a, 1.8) / 1.8
    r = raw['r'][sl].copy(); hl = raw['hl'][sl].copy(); vol = raw['vol'][sl].copy(); tr = raw['trades'][sl].copy(); lg = raw['large'][sl].copy()
    bf = np.where(vol > 0, raw['buy'][sl] / np.maximum(vol, 1e-12), 0.5)
    if fam == 1:   # pump and dump: up phase (60%) then dump
        up = u < 0.6; sh = np.where(up, u / 0.6, 1 - (u - 0.6) / 0.4); sg = np.where(up, 1.0, -1.0)
        r += sg * np.where(up, 1.5, 2.0) * a * sig * (0.5 + sh); vol *= 1 + 3 * a * sh; tr *= 1 + 2 * a * sh; lg *= 1 + 3 * a * sh
        bf = np.clip(bf + sg * 0.2 * amin * sh, 0.02, 0.98); hl = np.maximum(hl * (1 + a * sh), 0)
    elif fam == 2:  # fake breakout: up then full reversal, spike near the turn
        n1 = max(1, int(0.4 * D)); n2 = max(1, D - n1); mu1 = 2.0 * a * sig; mu = np.where(k < n1, mu1, -mu1 * n1 / n2)
        bump = np.exp(-((u - 0.35) / 0.2) ** 2); r += mu; vol *= 1 + 2.5 * a * bump; tr *= 1 + 2.0 * a * bump; lg *= 1 + 2.5 * a * bump
        bf = np.clip(bf + np.where(k < n1, 0.2, -0.2) * amin, 0.02, 0.98); hl = hl * (1 + a * bump)
    elif fam == 3:  # abnormal volume burst, price almost unchanged
        bump = np.sin(np.pi * u); vol *= 1 + 5 * a * bump; tr *= 1 + 3 * a * bump; lg *= 1 + 5 * a * bump; r *= 1 + 0.5 * a * bump
        bf = np.clip(bf + rng.uniform(-0.05, 0.05, D) * bump, 0.02, 0.98); hl = hl * (1 + 0.5 * a * bump)
    elif fam == 4:  # persistent trade-flow imbalance
        bf = np.clip(bf + sign * min(0.25 * a, 0.45) * ramp, 0.02, 0.98); vol *= 1 + 0.3 * a * ramp; tr *= 1 + 0.2 * a * ramp; r += sign * 0.3 * a * sig * ramp
    elif fam == 5:  # wash-like: more volume and trades, balanced flow, compressed range
        vol *= 1 + 1.5 * a * ramp; tr *= 1 + 2.5 * a * ramp; lg *= 1 + 2.0 * a * ramp; hl = hl * (1 - 0.4 * amin * ramp); r = r * (1 - 0.5 * amin * ramp)
        bf = 0.5 + (bf - 0.5) * (1 - 0.8 * ramp)
    elif fam == 6:  # liquidity stress: wide range, jagged returns, noisy flow
        alt = np.where(k % 2 == 0, 1.0, -1.0); hl = hl * (1 + 3 * a * ramp); vol *= 1 + a * ramp; tr *= 1 + a * ramp; lg *= 1 + 1.5 * a * ramp
        r += alt * 1.5 * a * sig * ramp; bf = np.clip(bf + rng.uniform(-0.1, 0.1, D) * ramp, 0.02, 0.98)
    elif fam == 7:  # momentum ignition (new): accelerating one-sided ramp, then sharp reversal
        n1 = max(2, int(0.8 * D)); r1 = np.where(k < n1, (np.minimum(k, n1 - 1) / n1) ** 1.5, 1.0); rev = k >= n1
        r += np.where(rev, -sign * 3.0 * a * sig, sign * 1.0 * a * sig * r1); vol *= 1 + 3 * a * r1; tr *= 1 + 2 * a * r1; lg *= 1 + 3 * a * r1
        bf = np.clip(bf + np.where(rev, -sign * 0.2, sign * 0.25) * min(a, 1.6) / 1.6 * np.maximum(r1, 0.2), 0.02, 0.98); hl = hl * (1 + a * r1)
    elif fam == 8:  # slow accumulation (new): long, low-amplitude one-sided flow
        bf = np.clip(bf + sign * 0.12 * min(a, 1.8) * ramp, 0.02, 0.98); vol *= 1 + 0.4 * a * ramp; tr *= 1 + 0.3 * a * ramp; lg *= 1 + 0.5 * a * ramp
        r += sign * 0.3 * a * sig * ramp
    vol = np.maximum(vol, 0); tr = np.where(vol > 0, np.maximum(1.0, np.round(tr)), 0.0); lg = np.minimum(np.round(np.maximum(lg, 0)), tr)
    hl = np.maximum(hl, np.abs(r) * 1.1)
    raw['r'][sl] = r; raw['hl'][sl] = hl; raw['vol'][sl] = vol; raw['buy'][sl] = np.clip(bf, 0, 1) * vol; raw['trades'][sl] = tr; raw['large'][sl] = lg


def place_events(lo, hi, fams, rng, max_tries=20000):
    """events as (t0, D, fam) inside [lo, hi); at least GAP + TAIL bars apart; starts >= lo + LOOK."""
    taken = []; ev = []
    for fam in fams:
        d0, d1 = DUR[fam]; D = int(rng.integers(d0, d1 + 1)); ok = False
        for _ in range(max_tries):
            t0 = int(rng.integers(lo + LOOK, hi - D - TAIL))
            if all(t0 + D + TAIL + GAP <= s or e + TAIL + GAP <= t0 for s, e in taken):
                taken.append((t0, t0 + D)); ev.append((t0, D, fam)); ok = True; break
        if not ok: break
    return sorted(ev)


def amp(rng): return float(np.exp(rng.uniform(np.log(0.5), np.log(2.0))))


def build_replica(m, symbol, lo, hi, fams, seed):
    rng = np.random.default_rng(seed); raw = raw_from_matrix(m); base = {k: v.copy() for k, v in raw.items()}
    order = list(fams); rng.shuffle(order); ev = place_events(lo, hi, order, rng)
    lab = np.zeros(len(m), dtype=np.int16); eid = np.full(len(m), -1, dtype=np.int32); rows = []
    for i, (t0, D, fam) in enumerate(ev):
        a = amp(rng)
        # event is applied on the ORIGINAL baselines: events are far apart, so the previous LOOK bars of a later event are untouched
        apply_event(raw, fam, t0, D, a, rng); lab[t0:t0 + D] = fam; eid[t0:t0 + D] = i; lab[t0 + D:t0 + D + TAIL] = -1
        rows.append({'event': i, 't0': t0, 'dur': D, 'family': fam, 'name': FAMS[fam], 'amp': a})
    f = features_from_raw(raw, symbol); f['label'] = lab; f['event_id'] = eid
    f['volume'] = raw['vol']; f['trades_count'] = raw['trades']
    return f, pd.DataFrame(rows), base


def capacity(lo, hi, fams):
    md = np.mean([np.mean(DUR[f]) for f in fams]); return max(1, int(0.5 * (hi - lo - LOOK) / (md + TAIL + GAP)))


def build_set(m, symbol, lo, hi, total, fams, seed):
    """pooled replicas; the family mix is equal over `fams`."""
    cap = capacity(lo, hi, fams); nrep = int(np.ceil(total / cap)); per = int(np.ceil(total / nrep)); frames = []; events = []
    for r in range(nrep):
        rng = np.random.default_rng(seed * 1000 + r)
        fl = [fams[i % len(fams)] for i in range(per)]
        f, ev, _ = build_replica(m, symbol, lo, hi, fl, seed * 1000 + r + 7)
        f = f.copy(); f['replica'] = r; f['bar'] = np.arange(len(f)); f['in_seg'] = (f['bar'] >= lo) & (f['bar'] < hi); frames.append(f)
        ev['replica'] = r; events.append(ev)
    return pd.concat(frames), pd.concat(events, ignore_index=True), {'replicas': nrep, 'events_per_replica': per, 'capacity': cap}


def sha(path, n=12):
    h = hashlib.sha256(); h.update(open(path, 'rb').read()); return h.hexdigest()[:n]


def unit_tests():
    rng = np.random.default_rng(0); n = 6000; idx = pd.date_range('2024-05-01', periods=n, freq='1min', tz='UTC')
    vol = rng.gamma(2, 3, n) + 0.5; buy = vol * np.clip(rng.normal(0.5, 0.1, n), 0.05, 0.95); tr = np.maximum(1, np.round(vol * rng.uniform(40, 80, n))); lg = np.round(vol * rng.uniform(1, 3, n))
    r = rng.normal(0, 3e-4, n); r[0] = 0; hl = np.abs(r) * 1.5 + rng.uniform(1e-4, 4e-4, n)
    m = pd.DataFrame({'volume': vol, 'trades_count': tr, 'large_trade_count': lg, 'log_returns': r, 'hl_spread_ratio': hl,
                      'trade_flow_imbalance': (2 * buy - vol) / (vol + 1e-8)}, index=idx)
    m['rush_order_density'] = tr / (vol + 1e-8)
    s = pd.Series(vol); mu = s.shift(1).rolling(15).mean().to_numpy(); sd = s.shift(1).rolling(15).std(ddof=1).to_numpy()
    va = np.zeros(n); va[15:] = (vol[15:] - mu[15:]) / np.maximum(sd[15:], FLOOR['BTCUSDT']); m['volume_acceleration'] = va
    res = {}
    f0 = features_from_raw(raw_from_matrix(m), 'BTCUSDT')
    res['U01 contract recompute reproduces stored features'] = all(np.allclose(f0[c], m[c], rtol=1e-9, atol=1e-12) for c in FEATS)
    f, ev, base = build_replica(m, 'BTCUSDT', 0, n, [1, 2, 3, 4, 5, 6, 7, 8] * 3, 5)
    res['U02 all 24 events placed'] = len(ev) == 24 and set(ev.family) == set(range(1, 9))
    d = ev.sort_values('t0'); gaps_ok = all(d.t0.iloc[i + 1] >= d.t0.iloc[i] + d.dur.iloc[i] + TAIL + GAP for i in range(len(d) - 1))
    res['U03 events >= GAP+TAIL apart, durations inside the declared range'] = gaps_ok and all(DUR[r_.family][0] <= r_.dur <= DUR[r_.family][1] for r_ in d.itertuples())
    chg = np.zeros(n, bool)
    for c in FEATS:
        if c == 'volume_acceleration': continue
        chg |= f[c].to_numpy() != f0[c].to_numpy()
    inside = f.label.to_numpy() > 0
    res['U04 raw-driven features change only inside events'] = not chg[~inside].any()
    dv = f.volume_acceleration.to_numpy() != f0.volume_acceleration.to_numpy()
    allowed = (f.label.to_numpy() != 0)
    res['U05 volume_acceleration changes only inside events and the 15-bar tail'] = not dv[~allowed].any()
    res['U06 tail bars labelled -1 and exactly 15 per event'] = (f.label == -1).sum() == 15 * len(ev) or (f.label == -1).sum() <= 15 * len(ev)
    res['U07 counts valid (large <= trades, volume >= 0, buy <= volume, hl >= |r|)'] = bool((f.large_trade_count <= f.trades_count + 1e-9).all() and (f.volume >= 0).all() and (f.hl_spread_ratio >= f.log_returns.abs() - 1e-12).all())
    f2, ev2, _ = build_replica(m, 'BTCUSDT', 0, n, [1, 2, 3, 4, 5, 6, 7, 8] * 3, 5)
    res['U08 same seed gives identical output'] = f.equals(f2) and ev.equals(ev2)
    res['U09 amplitudes inside 0.5..2.0'] = bool(ev.amp.between(0.5, 2.0).all())
    big, evs, info = build_set(m, 'BTCUSDT', 0, n, 60, OLD_FAMS, 3)
    res['U10 pooled set: requested events, replicas, no new families'] = len(evs) >= 60 and evs.family.isin(OLD_FAMS).all() and info['replicas'] >= 1
    # each family leaves a signature in the direction it should (median over its events)
    sig = {}
    for fam in range(1, 9):
        sub = ev[ev.family == fam]
        if len(sub) == 0: continue
        ratios = [f.volume.iloc[r_.t0:r_.t0 + r_.dur].mean() / m.volume.iloc[r_.t0:r_.t0 + r_.dur].mean() for r_ in sub.itertuples()]
        sig[fam] = float(np.median(ratios))
    res['U11 volume rises in families 1,2,3,5,7,8 (median ratio > 1)'] = all(sig[k] > 1.0 for k in [1, 2, 3, 5, 7, 8])
    return res, sig

''')

import importlib, sys; sys.path.insert(0, '.'); import l3v3_lib as L; importlib.reload(L)
LIBSHA = sha_file('l3v3_lib.py'); (OUT/'l3v3_lib.py').write_text(Path('l3v3_lib.py').read_text())
res, sig = L.unit_tests()
for k, v in res.items(): say(('PASS ' if v else 'FAIL ') + k)
assert all(res.values()), 'unit tests failed'
say(f'generator library sha256 {LIBSHA}; median event/bar volume ratio per family on the toy: ' + str({k: round(v, 2) for k, v in sig.items()}))

PASS U01 contract recompute reproduces stored features
PASS U02 all 24 events placed
PASS U03 events >= GAP+TAIL apart, durations inside the declared range
PASS U04 raw-driven features change only inside events
PASS U05 volume_acceleration changes only inside events and the 15-bar tail
PASS U06 tail bars labelled -1 and exactly 15 per event
PASS U07 counts valid (large <= trades, volume >= 0, buy <= volume, hl >= |r|)
PASS U08 same seed gives identical output
PASS U09 amplitudes inside 0.5..2.0
PASS U10 pooled set: requested events, replicas, no new families
PASS U11 volume rises in families 1,2,3,5,7,8 (median ratio > 1)
generator library sha256 591da6b080f1293c; median event/bar volume ratio per family on the toy: {1: 2.34, 2: 2.12, 3: 3.95, 4: 1.17, 5: 2.54, 6: 2.16, 7: 1.93, 8: 1.22}


## 3 · Load Layer 1 and Aug–Oct, reconstruction test

In [ ]:
M = {}; PATHS = {}
for s in SYMBOLS:
    p1 = find1(f'{s.lower()}_l1_feature_matrix_90d.parquet'); p2 = BASE/'layer1_aug_oct_v1'/f'{s.lower()}_l1_feature_matrix_aug_oct_2024.parquet'
    a = pd.read_parquet(p1)
    if 'datetime' in a.columns: a = a.set_index('datetime')
    a.index = pd.to_datetime(a.index, utc=True); b = pd.read_parquet(p2); b.index = pd.to_datetime(b.index, utc=True)
    assert len(a) == 129600 and len(b) == 132480, (len(a), len(b))
    M[s] = {'L1': a, 'AO': b}; PATHS[s] = {'L1': (p1, sha_file(p1)), 'AO': (str(p2), sha_file(p2))}
    # the warm-up contract: first 15 bars of each run are masked to 0 (Layer 1 stores bars 1..14 unmasked)
    for k in ('L1', 'AO'):
        d = M[s][k].copy(); d.loc[d.index[:15], 'volume_acceleration'] = 0.0; M[s][k] = d
    say(f'{s}: Layer 1 {a.index[0]} .. {a.index[-1]} ({len(a)} bars), Aug-Oct {b.index[0]} .. {b.index[-1]} ({len(b)} bars); input sha256 {PATHS[s]["L1"][1]} / {PATHS[s]["AO"][1]}')
    # reconstruction test on REAL data: raw quantities -> contract features equal the stored features
    for k in ('L1', 'AO'):
        d = M[s][k]; f = L.features_from_raw(L.raw_from_matrix(d), s); worst = {}
        for c in L.FEATS:
            x, y = f[c].to_numpy()[15:], d[c].to_numpy()[15:]; worst[c] = float(np.max(np.abs(x - y) / np.maximum(1.0, np.abs(y))))
        say(f'  reconstruction {k}: max relative difference per feature ' + str({c: float(f'{v:.1e}') for c, v in worst.items()}))
        assert max(worst.values()) < 5e-3, 'reconstruction from raw bar quantities does not reproduce the stored features'

BTCUSDT: Layer 1 2024-05-01 00:00:00+00:00 .. 2024-07-29 23:59:00+00:00 (129600 bars), Aug-Oct 2024-08-01 00:00:00+00:00 .. 2024-10-31 23:59:00+00:00 (132480 bars); input sha256 fdd121c8180cc895 / ea19ff4d8a834569
  reconstruction L1: max relative difference per feature {'log_returns': 0.0, 'trade_flow_imbalance': 2.2e-16, 'hl_spread_ratio': 0.0, 'rush_order_density': 0.0, 'volume_acceleration': 8e-06, 'large_trade_count': 0.0}
  reconstruction AO: max relative difference per feature {'log_returns': 0.0, 'trade_flow_imbalance': 2.2e-16, 'hl_spread_ratio': 0.0, 'rush_order_density': 0.0, 'volume_acceleration': 1.1e-11, 'large_trade_count': 0.0}
ETHUSDT: Layer 1 2024-05-01 00:00:00+00:00 .. 2024-07-29 23:59:00+00:00 (129600 bars), Aug-Oct 2024-08-01 00:00:00+00:00 .. 2024-10-31 23:59:00+00:00 (132480 bars); input sha256 20b915e3618d05e5 / aee064b697b8bde3
  reconstruction L1: max relative difference per feature {'log_returns': 0.0, 'trade_flow_imbalance': 2.2e-16, 'hl_spread_ratio': 0.0,

## 4 · Build the injected sets

In [ ]:
DAY = 1440
SEG = {'TRAIN': ('L1', 0, 90720, 600, L.OLD_FAMS, 11), 'VAL': ('L1', 90720, 129600, 300, L.OLD_FAMS + L.NEW_FAMS, 22),
       'CAL': ('AO', DAY, 31*DAY, 200, L.OLD_FAMS + L.NEW_FAMS, 33), 'TEST': ('AO', 31*DAY, 132480, 300, L.OLD_FAMS + L.NEW_FAMS, 44)}
INFO = {}; OUTFILES = []
for s in SYMBOLS:
    for name, (src, lo, hi, total, fams, seed) in SEG.items():
        seed2 = seed + (0 if s == 'BTCUSDT' else 5)
        big, ev, info = L.build_set(M[s][src], s, lo, hi, total, fams, seed2)
        assert name != 'TRAIN' or not ev.family.isin(L.NEW_FAMS).any(), 'new families must not appear in TRAIN-3'
        p = OUT/f'l3v3_{s}_{name}.parquet'; big.astype({c: 'float32' for c in ['log_returns','trade_flow_imbalance','hl_spread_ratio','rush_order_density','volume_acceleration','large_trade_count','volume','trades_count']}).to_parquet(p)
        ev.to_csv(OUT/f'l3v3_{s}_{name}_events.csv', index=False); OUTFILES += [p, OUT/f'l3v3_{s}_{name}_events.csv']
        INFO[(s, name)] = info | {'events': int(len(ev)), 'segment': [lo, hi], 'source': src, 'seed': seed2, 'rows': int(len(big))}
        say(f'{s} {name}: events {len(ev)} ({info["replicas"]} replica(s) x {info["events_per_replica"]}), per family ' + str(ev.family.value_counts().sort_index().to_dict()) + f', rows {len(big):,}')

BTCUSDT TRAIN: events 600 (2 replica(s) x 300), per family {1: 100, 2: 100, 3: 100, 4: 100, 5: 100, 6: 100}, rows 259,200
BTCUSDT VAL: events 300 (2 replica(s) x 150), per family {1: 38, 2: 38, 3: 38, 4: 38, 5: 38, 6: 38, 7: 36, 8: 36}, rows 259,200
BTCUSDT CAL: events 200 (1 replica(s) x 200), per family {1: 25, 2: 25, 3: 25, 4: 25, 5: 25, 6: 25, 7: 25, 8: 25}, rows 132,480
BTCUSDT TEST: events 300 (1 replica(s) x 300), per family {1: 38, 2: 38, 3: 38, 4: 38, 5: 37, 6: 37, 7: 37, 8: 37}, rows 132,480
ETHUSDT TRAIN: events 600 (2 replica(s) x 300), per family {1: 100, 2: 100, 3: 100, 4: 100, 5: 100, 6: 100}, rows 259,200
ETHUSDT VAL: events 300 (2 replica(s) x 150), per family {1: 38, 2: 38, 3: 38, 4: 38, 5: 38, 6: 38, 7: 36, 8: 36}, rows 259,200
ETHUSDT CAL: events 200 (1 replica(s) x 200), per family {1: 25, 2: 25, 3: 25, 4: 25, 5: 25, 6: 25, 7: 25, 8: 25}, rows 132,480
ETHUSDT TEST: events 300 (1 replica(s) x 300), per family {1: 38, 2: 38, 3: 38, 4: 38, 5: 37, 6: 37, 7: 37, 8: 37},

## 5 · Detectability sanity check

In [ ]:
# detectability sanity check with a simple robust-z score (not a model): how often does an event reach the 99.9th percentile of genuine scores?
from numpy.lib.stride_tricks import sliding_window_view
def rz_score(f, med, scl):
    x = f[L.FEATS].copy(); x['rush_order_density'] = np.log1p(x['rush_order_density']); x['large_trade_count'] = np.log1p(x['large_trade_count'])
    z = (x - med) / scl; return np.abs(z).clip(upper=20).max(axis=1).to_numpy()
rows = []
for s in SYMBOLS:
    g = M[s]['L1'].iloc[15:90720][L.FEATS].copy(); g['rush_order_density'] = np.log1p(g['rush_order_density']); g['large_trade_count'] = np.log1p(g['large_trade_count'])
    med = g.median(); scl = (g.quantile(.75) - g.quantile(.25)) / 1.349
    for name in ['VAL', 'TEST']:
        f = pd.read_parquet(OUT/f'l3v3_{s}_{name}.parquet'); f = f[f.replica == 0].reset_index(drop=True)
        ev = pd.read_csv(OUT/f'l3v3_{s}_{name}_events.csv'); ev = ev[ev.replica == 0]
        sc = rz_score(f.rename(columns={}), med, scl); clean = (f.label.to_numpy() == 0) & f.in_seg.to_numpy(); thr = np.quantile(sc[clean], 0.999)
        for fam, sub in ev.groupby('family'):
            hit = [sc[r.t0:r.t0 + r.dur].max() > thr for r in sub.itertuples()]
            rows.append((s, name, int(fam), L.FAMS[int(fam)], len(sub), float(np.mean(hit))))
df = pd.DataFrame(rows, columns=['symbol','set','family','name','events','share reaching p99.9 of genuine robust-z'])
say(df.pivot_table(index=['family','name'], columns=['symbol','set'], values='share reaching p99.9 of genuine robust-z').round(2).to_string())
mean_hit = df.iloc[:, -1].mean(); say(f'mean share over families/sets: {mean_hit:.2f}')
if mean_hit > 0.97: say('WARNING: injections are almost trivially detectable by a robust-z score (check amplitudes before training)')
if mean_hit < 0.10: say('WARNING: injections are almost undetectable (check amplitudes before training)')

symbol                           BTCUSDT      ETHUSDT     
set                                 TEST  VAL    TEST  VAL
family name                                               
1      pump_and_dump_like            0.0  0.0     0.0  0.0
2      fake_breakout_like            0.0  0.0     0.0  0.0
3      abnormal_volume_burst         0.0  0.0     0.0  0.0
4      trade_flow_imbalance          0.0  0.0     0.0  0.0
5      wash_like_aggregate_proxy     0.0  0.0     0.0  0.0
6      liquidity_stress_proxy        0.0  0.0     0.0  0.0
7      momentum_ignition_like        0.0  0.0     0.0  0.0
8      slow_accumulation_like        0.0  0.0     0.0  0.0
mean share over families/sets: 0.00


## 6 · Manifest and share block

In [ ]:
man = {'protocol_sha256': ph, 'lib_sha256_16': LIBSHA, 'inputs': {s: {k: list(v) for k, v in PATHS[s].items()} for s in SYMBOLS},
       'sets': {f'{k[0]}|{k[1]}': v for k, v in INFO.items()}, 'files': {p.name: sha_file(p) for p in OUTFILES}, 'created': datetime.datetime.utcnow().isoformat() + 'Z'}
(OUT/'manifest.json').write_text(json.dumps(man, indent=1, default=str)); block = '\n'.join(LINES); (OUT/'l3v3_share_block.txt').write_text(block)
print('\n===== SHARE BLOCK (copy everything below, or attach layer3_v3/l3v3_share_block.txt) =====\n' + block[-7000:])


===== SHARE BLOCK (copy everything below, or attach layer3_v3/l3v3_share_block.txt) =====
PROTOCOL FROZEN now; sha256 443f99d5faf5929f9b58d510e5aaf955a65359ef577c9590c54246d3d7254e9e
PASS U01 contract recompute reproduces stored features
PASS U02 all 24 events placed
PASS U03 events >= GAP+TAIL apart, durations inside the declared range
PASS U04 raw-driven features change only inside events
PASS U05 volume_acceleration changes only inside events and the 15-bar tail
PASS U06 tail bars labelled -1 and exactly 15 per event
PASS U07 counts valid (large <= trades, volume >= 0, buy <= volume, hl >= |r|)
PASS U08 same seed gives identical output
PASS U09 amplitudes inside 0.5..2.0
PASS U10 pooled set: requested events, replicas, no new families
PASS U11 volume rises in families 1,2,3,5,7,8 (median ratio > 1)
generator library sha256 591da6b080f1293c; median event/bar volume ratio per family on the toy: {1: 2.34, 2: 2.12, 3: 3.95, 4: 1.17, 5: 2.54, 6: 2.16, 7: 1.93, 8: 1.22}
BTCUSDT: Layer 1 